# RF-DETR Large 74-Class Hidden45 Colab CUDA 5-Fold Training (v1.8+)

## Goal

Train the latest 74-class pill dataset with RF-DETR Large using an up-to-date `rfdetr>=1.8.0` install, then run 5-fold training from the same notebook.

Default path:

```text
Google Drive dataset tar.gz
  -> /content/rfdetr_colab/datasets/rfdetr_dataset_74_hidden45_canvas_balanced
  -> build 5 folds with class-0 dummy placeholder
  -> RF-DETR Large train/valid dry-run per fold
  -> CUDA full train/resume per fold
  -> mAP@0.75 checkpoint selection per fold
```

The notebook intentionally does not do final submission postprocessing. Use the local inference/postprocess notebook after checkpoints are produced. This keeps Colab focused on expensive training and checkpoint persistence.


## Colab Checklist

1. Set `Runtime > Change runtime type > GPU`. RF-DETR Large is much heavier than nano/medium; L4/A100 is safer than T4.
2. The notebook uses local cache first. If `DATASET_DIR` already contains `train/valid/_annotations.coco.json`, archive download/extract is skipped.
3. In Colab, the Drive archive is copied/downloaded once into `/content/rfdetr_colab/archives` and extracted under `/content/rfdetr_colab/datasets`, so training and fold building read local files instead of repeatedly reading Drive.
4. For a local run, set `DATASET_DIR` to an existing extracted dataset folder and `RUN_EXTRACT_DATASET=False`; no archive is needed.
5. Run once with `RUN_DRY_RUN=True` and `RUN_FULL_TRAIN=False`; this builds/validates all fold configs without training.
6. If dry-run passes, set `RUN_FULL_TRAIN=True`. This copy is configured for `FOLD_INDICES = [4]` only, so it can run fold4 on Colab while local MPS output remains separate.
7. 5-fold validation excludes synthetic `canvas__` images by default; canvas samples remain train-only unless `VALID_INCLUDE_CANVAS=True`.
8. Class `0` is the background/no-object dummy category. It is included in `categories` and `label_map_74.json`; background learning comes from unmatched/no-object regions, so it must have zero bbox annotations. Real pill classes use labels `1..74`.
9. Test submission is deliberately left to local inference/postprocessing after checkpoints are ready.

Default dataset archive is the canvas-balanced package because it better matches the full test-image scale while keeping validation non-synthetic.


In [ ]:
# =========================
# Parameters - edit here
# =========================
from pathlib import Path

RUN_MOUNT_DRIVE = True
RUN_CLONE_OR_UPDATE_REPO = True
RUN_INSTALL_DEPS = True
RUN_EXTRACT_DATASET = True
FORCE_REEXTRACT_DATASET = False
USE_READY_DATASET_DIR_IF_EXISTS = True
CACHE_ARCHIVE_BEFORE_EXTRACT = True
RUN_BUILD_5FOLD_DATASET = True
FORCE_REBUILD_5FOLD_DATASET = False
RUN_DRY_RUN = False
RUN_IMPORT_SMOKE_TEST = True
RUN_FULL_TRAIN = True
RUN_FINALIZE_ONLY = False
RUN_SHOW_TENSORBOARD = False
RUN_TEST_INFERENCE = False  # keep False here; do submission/postprocess locally after training
REQUIRE_DRIVE_FOR_CHECKPOINTS = True

RFDETR_MIN_VERSION = '1.8.0'

TEAM_REPO_URL = 'https://github.com/kim-tae-yoon-0718/ai12-team01.git'
TEAM_REPO_BRANCH = 'model/rfdetr-basic74-45img-filled'

WORK_ROOT = Path('/content/rfdetr_colab')
REPO_DIR = WORK_ROOT / 'ai12-team01'
RFDETR_DIR = REPO_DIR / 'RF_DETR_split_ver'

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/ai12-level1-project')
DATASET_ARCHIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1eZp-iULcZhw30h-LExPvP-WRbnXtLzjv'
DATASET_ARCHIVE_FILE_ID = '1_DFR_eeEQVCnkF0NCYvIG6ESZFgo3rGa'
DATASET_ARCHIVE_FILE_URL = 'https://drive.google.com/file/d/1_DFR_eeEQVCnkF0NCYvIG6ESZFgo3rGa/view'
DATASET_ARCHIVE_DIR = DRIVE_PROJECT_DIR / 'dataset_74_hidden45_latest_20260706'
DATASET_ARCHIVE_NAME = 'dataset_74_hidden45_canvas_balanced_train_valid_20260706.tar.gz'
SCAN_MOUNTED_DRIVE_FOR_ARCHIVE = False  # keep False to avoid slow /content/drive recursive scans in Colab
ARCHIVE_CACHE_DIR = WORK_ROOT / 'archives'
DATASET_EXTRACT_DIR = WORK_ROOT / 'datasets'
# Local/cache dataset directory. For local runs, point this at an already extracted dataset and set RUN_EXTRACT_DATASET=False.
DATASET_DIR = DATASET_EXTRACT_DIR / 'rfdetr_dataset_74_hidden45_canvas_balanced'
FOLDED_DATASET_DIR = DATASET_EXTRACT_DIR / 'rfdetr_dataset_74_hidden45_canvas_balanced_5fold_cls0'

CONFIG_STEM = 'config_74_hidden45_colab_cuda_rfdetr_large_v18_5fold'
BASE_CONFIG_NAME = 'config_74_hidden45_canvas_balanced.yaml'
MODEL_VARIANT = 'large'  # nano | small | medium | base | large

# Full 5-fold run. For a quick test, change this to [0].
NUM_FOLDS = 5
FOLD_INDICES = [4]
ADD_CLASS0_PLACEHOLDER = True
CLASS0_NAME = 'background_dummy_placeholder'
VALID_INCLUDE_CANVAS = False  # keep synthetic canvas images out of validation; train may still use them

BASE_MODEL_TAG = 'large_74_hidden45_canvas_balanced_5fold_cls0_colab_cuda_rfdetr_v18plus'
ENABLE_TRAIN_AUGMENTATION = True
AUGMENTATION_NAME = 'aug_scale150_rot90_v1'
MODEL_TAG = f'{BASE_MODEL_TAG}_{AUGMENTATION_NAME}' if ENABLE_TRAIN_AUGMENTATION else BASE_MODEL_TAG

# Same effective batch as the previous runs: 1 x 16 = 16. Large starts at micro-batch 1 for OOM safety.
# Early stopping is validation/eval based, not mini-batch based: eval_interval=1 means patience counts epochs.
# One epoch is roughly 2.3k train images per fold, so patience=1 is the closest match to a ~2k-image no-improvement window.
EPOCHS = 100
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 1
EARLY_STOPPING_MIN_DELTA = 0.001
EARLY_STOPPING_USE_EMA = True
NUM_WORKERS = 2       # raise to 4 only if Colab runtime and Drive I/O are stable
PIN_MEMORY = True
TRAIN_EPOCHS_OVERRIDE = None  # keep None for config-controlled epochs

# Online train-time augmentation. Flip is intentionally excluded because mirrored pill imprints are unrealistic.
AUG_SCALE_RANGE = [0.85, 1.50]
AUG_TRANSLATE_PERCENT = [-0.04, 0.04]
AUG_ROTATE_LIMIT_DEGREES = 90
AUG_AFFINE_P = 0.35
AUG_BRIGHTNESS_LIMIT = 0.08
AUG_CONTRAST_LIMIT = 0.08
AUG_COLOR_P = 0.25
ENABLE_LIGHT_AUGMENTATION = ENABLE_TRAIN_AUGMENTATION
ENABLE_MULTI_SCALE = False

OUTPUT_ROOT = DRIVE_PROJECT_DIR / 'rfdetr_outputs'
BACKUP_DIR = OUTPUT_ROOT / 'best'


In [ ]:
# Imports and small helpers
import json
import os
import random
import selectors
import shutil
import subprocess
import sys
import tarfile
import time
from collections import Counter
from pathlib import Path

import pandas as pd


def _print_training_artifact_status():
    output_root = globals().get('OUTPUT_ROOT')
    model_tag = globals().get('MODEL_TAG')
    if output_root is None or model_tag is None:
        print('[status] OUTPUT_ROOT/MODEL_TAG not defined yet.', flush=True)
        return

    output_dir = Path(output_root) / str(model_tag)
    print(f'[status] output_dir={output_dir} exists={output_dir.exists()}', flush=True)
    if not output_dir.exists():
        return

    metrics_csv = output_dir / 'metrics.csv'
    if metrics_csv.exists():
        try:
            tail = metrics_csv.read_text(encoding='utf-8', errors='replace').strip().splitlines()[-3:]
            print('[status] metrics.csv tail:', flush=True)
            for line in tail:
                print('  ' + line, flush=True)
        except Exception as exc:
            print(f'[status] could not read metrics.csv: {exc}', flush=True)
    else:
        print('[status] metrics.csv not created yet.', flush=True)

    checkpoints = sorted(output_dir.glob('checkpoint_*'), key=lambda path: path.stat().st_mtime if path.exists() else 0)
    if checkpoints:
        latest = checkpoints[-1]
        print(f'[status] latest checkpoint={latest.name} size_MB={latest.stat().st_size / (1024 * 1024):.1f}', flush=True)
    else:
        print('[status] no checkpoint files yet.', flush=True)


def run(cmd, cwd=None, env=None, heartbeat_seconds=30):
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)
    merged_env.setdefault('PYTHONUNBUFFERED', '1')
    merged_env.setdefault('PYTHONFAULTHANDLER', '1')
    merged_env.setdefault('WANDB_DISABLED', 'true')
    merged_env.setdefault('WANDB_MODE', 'disabled')

    cmd_list = list(map(str, cmd))
    printable_cmd = ' '.join(cmd_list)
    print('run:', printable_cmd, flush=True)
    print('cwd:', cwd or Path.cwd(), flush=True)
    print('started_at:', time.strftime('%Y-%m-%d %H:%M:%S'), flush=True)

    process = subprocess.Popen(
        cmd_list,
        cwd=cwd,
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    selector = selectors.DefaultSelector()
    selector.register(process.stdout, selectors.EVENT_READ)
    start_time = time.monotonic()
    last_output = start_time

    try:
        while True:
            events = selector.select(timeout=5)
            for key, _ in events:
                chunk = os.read(key.fileobj.fileno(), 8192)
                if not chunk:
                    continue
                print(chunk.decode('utf-8', errors='replace'), end='', flush=True)
                last_output = time.monotonic()

            returncode = process.poll()
            if returncode is not None:
                while True:
                    chunk = process.stdout.read(8192)
                    if not chunk:
                        break
                    print(chunk.decode('utf-8', errors='replace'), end='', flush=True)
                break

            now = time.monotonic()
            if now - last_output >= heartbeat_seconds:
                elapsed = int(now - start_time)
                print('\n' + f'[still running] elapsed={elapsed}s, no subprocess output for {int(now - last_output)}s', flush=True)
                _print_training_artifact_status()
                last_output = now
    finally:
        try:
            selector.unregister(process.stdout)
        except Exception:
            pass
        if process.stdout:
            process.stdout.close()

    print('finished_at:', time.strftime('%Y-%m-%d %H:%M:%S'), 'returncode:', returncode, flush=True)
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, cmd_list)
    return subprocess.CompletedProcess(cmd_list, returncode)


def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


seed_everything(42)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print('WORK_ROOT:', WORK_ROOT)
print('Python:', sys.executable)


In [ ]:
# Mount Google Drive and make checkpoint persistence explicit
if RUN_MOUNT_DRIVE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive mount skipped or unavailable:', exc)
else:
    print('Skipping Drive mount.')

if REQUIRE_DRIVE_FOR_CHECKPOINTS and not Path('/content/drive/MyDrive').exists():
    raise RuntimeError(
        'Google Drive is not mounted, so checkpoints would not be persistent. '
        'Set RUN_MOUNT_DRIVE=True and authorize Drive before training.'
    )

print('checkpoint/output root:', OUTPUT_ROOT)
print('backup root:', BACKUP_DIR)


In [ ]:
# Clone or update team repo
if RUN_CLONE_OR_UPDATE_REPO:
    if not REPO_DIR.exists():
        run(['git', 'clone', TEAM_REPO_URL, REPO_DIR])
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    checkout_cmd = ['git', 'checkout', TEAM_REPO_BRANCH]
    try:
        run(checkout_cmd, cwd=REPO_DIR)
    except subprocess.CalledProcessError:
        run(['git', 'checkout', '-B', TEAM_REPO_BRANCH, f'origin/{TEAM_REPO_BRANCH}'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', TEAM_REPO_BRANCH], cwd=REPO_DIR)
else:
    print('Skipping repo clone/update.')

if not RFDETR_DIR.exists():
    raise FileNotFoundError(f'RF_DETR_split_ver not found: {RFDETR_DIR}')
os.chdir(RFDETR_DIR)
print('cwd:', Path.cwd())


In [ ]:
# Install dependencies
if RUN_INSTALL_DEPS:
    # Colab already provides CUDA PyTorch. requirements.txt should usually be satisfied without replacing it.
    run([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO_DIR / 'requirements.txt'])
    run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', f'rfdetr[train,loggers]>={RFDETR_MIN_VERSION}', 'packaging'])
    run([sys.executable, '-m', 'pip', 'install', '-q', 'pycocotools', 'faster-coco-eval', 'opencv-python-headless', 'tqdm', 'gdown'])
else:
    print('Skipping dependency install.')


In [ ]:
# Runtime check
import importlib.metadata
import torch
from packaging.version import Version

try:
    import rfdetr
    rfdetr_version = importlib.metadata.version('rfdetr')
    rfdetr_status = f'ok ({rfdetr_version})'
except Exception as exc:
    rfdetr_version = None
    rfdetr_status = f'IMPORT FAILED: {exc}'

runtime_summary = {
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'rfdetr': rfdetr_status,
    'required_rfdetr_min_version': RFDETR_MIN_VERSION,
    'model_variant': MODEL_VARIANT,
}
print(json.dumps(runtime_summary, indent=2, ensure_ascii=False))
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available. Change Colab runtime type to GPU before training.')
if rfdetr_version is None or Version(rfdetr_version) < Version(RFDETR_MIN_VERSION):
    raise RuntimeError(f'rfdetr>={RFDETR_MIN_VERSION} is required, got {rfdetr_version}')
if not hasattr(rfdetr, 'RFDETRLarge'):
    raise RuntimeError('Installed rfdetr package does not expose RFDETRLarge.')


In [ ]:
# Locate/download archive only when local/cache dataset is not already ready
def dataset_ready(dataset_dir: Path) -> bool:
    return (dataset_dir / 'train' / '_annotations.coco.json').exists() and (dataset_dir / 'valid' / '_annotations.coco.json').exists()

ARCHIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
DATASET_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

if USE_READY_DATASET_DIR_IF_EXISTS and dataset_ready(DATASET_DIR) and not FORCE_REEXTRACT_DATASET:
    archive_path = None
    archive_source = 'skipped: DATASET_DIR is already ready'
    print('Using ready local/cache dataset:', DATASET_DIR)
    print('Archive download/extract skipped.')
elif not RUN_EXTRACT_DATASET:
    raise FileNotFoundError(
        f'DATASET_DIR is not ready and RUN_EXTRACT_DATASET=False: {DATASET_DIR}. '
        'For local runs, set DATASET_DIR to an existing extracted train/valid dataset.'
    )
else:
    archive_path = DATASET_ARCHIVE_DIR / DATASET_ARCHIVE_NAME
    archive_source = 'configured mounted Drive path'

    if not archive_path.exists():
        print('Mounted archive path not found:', archive_path)
        print('This is normal when the Drive folder is shared or not added as a My Drive shortcut.')

        candidates = []
        if SCAN_MOUNTED_DRIVE_FOR_ARCHIVE:
            print('Scanning /content/drive for the archive. This can be slow on large Drives.')
            search_root = Path('/content/drive')
            candidates = list(search_root.rglob(DATASET_ARCHIVE_NAME))[:20] if search_root.exists() else []
        else:
            print('Skipping recursive /content/drive scan; using cache or Drive file ID instead.')

        cached_archive = ARCHIVE_CACHE_DIR / DATASET_ARCHIVE_NAME
        if cached_archive.exists() and cached_archive.stat().st_size > 0 and not FORCE_REEXTRACT_DATASET:
            archive_path = cached_archive
            archive_source = 'existing /content archive cache'
            print('Using cached archive:', archive_path)
        elif candidates:
            archive_path = candidates[0]
            archive_source = 'found by scanning /content/drive'
            print('Using found archive:', archive_path)
        else:
            print('Downloading dataset archive from Drive file ID with gdown into /content cache.')
            print('Dataset folder URL:', DATASET_ARCHIVE_FOLDER_URL)
            print('Dataset file URL:', DATASET_ARCHIVE_FILE_URL)
            try:
                import gdown
            except Exception:
                run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
                import gdown
            archive_path = cached_archive
            gdown.download(id=DATASET_ARCHIVE_FILE_ID, output=str(archive_path), quiet=False, fuzzy=True)
            archive_source = 'gdown file id download to /content cache'

    if archive_path is None or not archive_path.exists() or archive_path.stat().st_size == 0:
        raise FileNotFoundError(f'Dataset archive not available after search/download: {archive_path}')

    # Avoid extracting directly from Drive. Copy the archive to /content cache first, then tar-read locally.
    cached_archive = ARCHIVE_CACHE_DIR / DATASET_ARCHIVE_NAME
    if CACHE_ARCHIVE_BEFORE_EXTRACT and archive_path != cached_archive:
        needs_copy = (not cached_archive.exists()) or cached_archive.stat().st_size != archive_path.stat().st_size or FORCE_REEXTRACT_DATASET
        if needs_copy:
            print('Copying archive to local /content cache before extraction...')
            shutil.copy2(archive_path, cached_archive)
        archive_path = cached_archive
        archive_source = archive_source + ' -> /content archive cache'

    print('archive source:', archive_source)
    print('archive:', archive_path)
    print('archive size GB:', round(archive_path.stat().st_size / 1024**3, 3))

    already_ready = dataset_ready(DATASET_DIR)
    if already_ready and not FORCE_REEXTRACT_DATASET:
        print('Dataset already extracted:', DATASET_DIR)
    else:
        if DATASET_DIR.exists():
            shutil.rmtree(DATASET_DIR)
        DATASET_DIR.mkdir(parents=True, exist_ok=True)
        with tarfile.open(archive_path, 'r:gz') as tar:
            tar.extractall(DATASET_DIR)
        print('Extracted to:', DATASET_DIR)

if not dataset_ready(DATASET_DIR):
    raise FileNotFoundError(f'Dataset is still not ready after cache/extract step: {DATASET_DIR}')


In [ ]:
# Dataset sanity checks
train_json = DATASET_DIR / 'train' / '_annotations.coco.json'
valid_json = DATASET_DIR / 'valid' / '_annotations.coco.json'
if not train_json.exists() or not valid_json.exists():
    raise FileNotFoundError(f'Missing train/valid annotations under {DATASET_DIR}')

summary_path = DATASET_DIR / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('dataset summary keys:', list(summary.keys()))
    if 'splits' in summary:
        display(pd.DataFrame(summary['splits']))
    if 'validation' in summary:
        print(json.dumps(summary['validation'].get('train_valid', summary['validation']), indent=2, ensure_ascii=False))
else:
    print('No summary.json found; continuing with COCO JSON checks.')

def load_coco(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

splits = {'train': load_coco(train_json), 'valid': load_coco(valid_json)}
rows = []
for split, data in splits.items():
    category_ids = sorted(int(c['id']) for c in data['categories'])
    ann_category_ids = sorted({int(a['category_id']) for a in data['annotations']})
    missing_files = []
    for image in data['images'][:]:
        if not (DATASET_DIR / split / image['file_name']).exists():
            missing_files.append(image['file_name'])
    rows.append({
        'split': split,
        'images': len(data['images']),
        'annotations': len(data['annotations']),
        'categories': len(category_ids),
        'ann_categories': len(ann_category_ids),
        'min_category_id': min(category_ids),
        'max_category_id': max(category_ids),
        'missing_files': len(missing_files),
    })
    if missing_files:
        raise FileNotFoundError(f'{split} has missing files, sample={missing_files[:5]}')

display(pd.DataFrame(rows))
category_counts = Counter(a['category_id'] for data in splits.values() for a in data['annotations'])
print('class annotation range:', min(category_counts.values()), 'to', max(category_counts.values()))
print('classes:', len(category_counts))


In [ ]:
# Build 5-fold COCO dataset with class-0 dummy placeholder
if RUN_BUILD_5FOLD_DATASET:
    import csv
    import json
    import os
    import random
    import re
    import shutil
    from collections import Counter, defaultdict
    from pathlib import Path

    import numpy as np
    from sklearn.model_selection import StratifiedGroupKFold

    ready_marker = FOLDED_DATASET_DIR / '_5fold_ready.json'
    if ready_marker.exists() and not FORCE_REBUILD_5FOLD_DATASET:
        print('5-fold dataset already exists:', FOLDED_DATASET_DIR)
        print(ready_marker.read_text(encoding='utf-8'))
    else:
        if FOLDED_DATASET_DIR.exists():
            shutil.rmtree(FOLDED_DATASET_DIR)
        FOLDED_DATASET_DIR.mkdir(parents=True, exist_ok=True)

        def load_split(split):
            ann_path = DATASET_DIR / split / '_annotations.coco.json'
            payload = json.loads(ann_path.read_text(encoding='utf-8'))
            images_by_id = {int(img['id']): img for img in payload.get('images', [])}
            anns_by_image = defaultdict(list)
            for ann in payload.get('annotations', []):
                anns_by_image[int(ann['image_id'])].append(ann)
            return payload, images_by_id, anns_by_image

        train_payload, train_images, train_anns = load_split('train')
        valid_payload, valid_images, valid_anns = load_split('valid')
        source_categories = {int(cat['id']): cat for cat in train_payload.get('categories', [])}
        source_categories.update({int(cat['id']): cat for cat in valid_payload.get('categories', [])})
        real_category_ids = sorted(cid for cid in source_categories if cid != 0)
        cat2label = {cid: i + 1 for i, cid in enumerate(real_category_ids)}
        label2cat = {label: cid for cid, label in cat2label.items()}

        def normalize_group_name(file_name):
            stem = Path(file_name).stem
            core = stem
            if core.startswith('canvas__'):
                core = core[len('canvas__'):]
            if '__' in core:
                core = core.split('__', 1)[1]
            match = re.search(r'(K-[0-9-]+_0_2_\d_2)_(\d+)(_000_200)', core)
            if match:
                return f'{match.group(1)}_ANGLE{match.group(3)}'
            return re.sub(r'_(70|75|90)_000_200', '_ANGLE_000_200', core)

        records = []
        used_output_names = Counter()
        for split, images_by_id, anns_by_image in [('train', train_images, train_anns), ('valid', valid_images, valid_anns)]:
            for image_id, image in images_by_id.items():
                anns = anns_by_image.get(image_id, [])
                if not anns:
                    continue
                src_path = DATASET_DIR / split / image['file_name']
                if not src_path.exists():
                    raise FileNotFoundError(src_path)
                base_output_name = f'{split}__{image["file_name"]}'
                used_output_names[base_output_name] += 1
                output_name = base_output_name
                if used_output_names[base_output_name] > 1:
                    output_name = f'{split}__dup{used_output_names[base_output_name]}__{image["file_name"]}'
                labels = sorted({int(ann['category_id']) for ann in anns})
                records.append({
                    'source_split': split,
                    'source_image_id': image_id,
                    'source_file_name': image['file_name'],
                    'output_file_name': output_name,
                    'width': int(image['width']),
                    'height': int(image['height']),
                    'src_path': src_path,
                    'is_canvas': image['file_name'].startswith('canvas__'),
                    'annotations': anns,
                    'category_ids': labels,
                    'group': normalize_group_name(image['file_name']),
                })

        if not records:
            raise RuntimeError('No annotated images found for 5-fold build.')

        freq = Counter(cid for rec in records for cid in rec['category_ids'])
        valid_candidate_indices = [
            i for i, rec in enumerate(records)
            if VALID_INCLUDE_CANVAS or not rec['is_canvas']
        ]
        if len(valid_candidate_indices) < NUM_FOLDS:
            raise RuntimeError(f'Not enough validation candidates for {NUM_FOLDS} folds: {len(valid_candidate_indices)}')

        file_indices = np.array(valid_candidate_indices)
        groups = np.array([records[i]['group'] for i in valid_candidate_indices])
        strat = np.array([min(records[i]['category_ids'], key=lambda cid: freq[cid]) for i in valid_candidate_indices])

        sgkf = StratifiedGroupKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)
        folds = list(sgkf.split(file_indices, strat, groups))
        print('5-fold validation candidates:', len(valid_candidate_indices), 'of records:', len(records), 'VALID_INCLUDE_CANVAS=', VALID_INCLUDE_CANVAS)

        def link_or_copy(src, dst):
            dst.parent.mkdir(parents=True, exist_ok=True)
            if dst.exists() or dst.is_symlink():
                dst.unlink()
            try:
                os.symlink(src, dst)
            except OSError:
                shutil.copy2(src, dst)

        def categories_for_fold():
            cats = []
            if ADD_CLASS0_PLACEHOLDER:
                cats.append({
                    'id': 0,
                    'name': CLASS0_NAME,
                    'supercategory': 'background',
                    'is_placeholder': True,
                    'used_in_annotations': False,
                    'note': 'Background/no-object dummy category. Keep bbox annotations at zero; never submit category 0.',
                })
            for cid in real_category_ids:
                src_cat = dict(source_categories[cid])
                label = cat2label[cid]
                cats.append({
                    'id': label,
                    'name': str(src_cat.get('name', cid)),
                    'supercategory': src_cat.get('supercategory', 'pill'),
                    'original_category_id': cid,
                    'source_name': src_cat.get('name', ''),
                })
            return cats

        fold_summaries = []
        category_rows = []
        if ADD_CLASS0_PLACEHOLDER:
            category_rows.append({
                'rfdetr_internal_label': 0,
                'coco_category_id': 0,
                'category_id': 0,
                'n_number': 'N00',
                'name': CLASS0_NAME,
                'is_placeholder': True,
            })
        source_mapping_path = DATASET_DIR / 'category_mapping.csv'
        source_mapping = {}
        if source_mapping_path.exists():
            with source_mapping_path.open('r', encoding='utf-8-sig', newline='') as f:
                for row in csv.DictReader(f):
                    source_mapping[int(row['category_id'])] = row
        for cid in real_category_ids:
            src_row = source_mapping.get(cid, {})
            category_rows.append({
                'rfdetr_internal_label': cat2label[cid],
                'coco_category_id': cat2label[cid],
                'category_id': cid,
                'n_number': src_row.get('n_number', ''),
                'name': src_row.get('name', source_categories[cid].get('name', '')),
                'is_placeholder': False,
            })

        all_indices = list(range(len(records)))
        for fold_idx, (_candidate_train_idx, candidate_valid_idx) in enumerate(folds):
            fold_dir = FOLDED_DATASET_DIR / f'fold{fold_idx}'
            valid_idx = [valid_candidate_indices[int(i)] for i in candidate_valid_idx]
            valid_groups = {records[i]['group'] for i in valid_idx}
            train_idx = [i for i in all_indices if records[i]['group'] not in valid_groups]
            train_groups = {records[i]['group'] for i in train_idx}
            leakage = sorted(train_groups & valid_groups)
            if leakage:
                raise RuntimeError(f'fold{fold_idx} group leakage detected: {leakage[:5]}')
            if not VALID_INCLUDE_CANVAS and any(records[i]['is_canvas'] for i in valid_idx):
                raise RuntimeError(f'fold{fold_idx} validation unexpectedly contains canvas images')

            for split_name, indices in [('train', train_idx), ('valid', valid_idx)]:
                split_dir = fold_dir / split_name
                split_dir.mkdir(parents=True, exist_ok=True)
                images = []
                annotations = []
                ann_id = 1
                for new_image_id, rec_index in enumerate(indices, start=1):
                    rec = records[int(rec_index)]
                    images.append({
                        'id': new_image_id,
                        'file_name': rec['output_file_name'],
                        'width': rec['width'],
                        'height': rec['height'],
                        'source_split': rec['source_split'],
                        'source_file_name': rec['source_file_name'],
                        'group_key': rec['group'],
                    })
                    link_or_copy(rec['src_path'], split_dir / rec['output_file_name'])
                    for ann in rec['annotations']:
                        original_cid = int(ann['category_id'])
                        if original_cid == 0:
                            raise RuntimeError('source annotation unexpectedly used category_id=0')
                        bbox = [float(v) for v in ann['bbox']]
                        annotations.append({
                            'id': ann_id,
                            'image_id': new_image_id,
                            'category_id': cat2label[original_cid],
                            'bbox': bbox,
                            'area': float(ann.get('area', bbox[2] * bbox[3])),
                            'iscrowd': int(ann.get('iscrowd', 0)),
                            'original_category_id': original_cid,
                            'source_annotation_id': ann.get('id', ''),
                        })
                        ann_id += 1
                payload = {
                    'info': {
                        'description': 'RF-DETR 74-class 5-fold dataset with class-0 background/no-object dummy category',
                        'class0_semantics': 'category_id=0 is the background/no-object dummy category; bbox annotations use labels 1..74.',
                    },
                    'images': images,
                    'annotations': annotations,
                    'categories': categories_for_fold(),
                }
                (split_dir / '_annotations.coco.json').write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')

            train_class_counts = Counter(
                cid for i in train_idx for cid in records[i]['category_ids']
            )
            valid_class_counts = Counter(
                cid for i in valid_idx for cid in records[i]['category_ids']
            )
            fold_summary = {
                'fold': fold_idx,
                'train_images': int(len(train_idx)),
                'valid_images': int(len(valid_idx)),
                'train_canvas_images': int(sum(records[i]['is_canvas'] for i in train_idx)),
                'valid_canvas_images': int(sum(records[i]['is_canvas'] for i in valid_idx)),
                'train_groups': int(len(train_groups)),
                'valid_groups': int(len(valid_groups)),
                'group_leakage': 0,
                'train_classes': int(len(train_class_counts)),
                'valid_classes': int(len(valid_class_counts)),
                'valid_min_class_annotations': int(min(valid_class_counts.values())) if valid_class_counts else 0,
                'valid_max_class_annotations': int(max(valid_class_counts.values())) if valid_class_counts else 0,
            }
            fold_summaries.append(fold_summary)
            print('fold summary:', fold_summary)

        label_map = {
            'class0': {
                'id': 0,
                'name': CLASS0_NAME,
                'is_placeholder': True,
                'used_in_annotations': False,
                'purpose': 'Represent the background/no-object dummy category; real pill classes use labels 1..74.',
                'submission_rule': 'Drop predictions mapped to category_id 0 during local postprocess.',
            },
            'cat2label': {str(k): int(v) for k, v in cat2label.items()},
            'label2cat': {str(k): int(v) for k, v in label2cat.items()},
            'num_real_classes': len(real_category_ids),
            'num_coco_categories_including_class0': len(real_category_ids) + int(ADD_CLASS0_PLACEHOLDER),
        }
        (FOLDED_DATASET_DIR / 'label_map_74.json').write_text(json.dumps(label_map, ensure_ascii=False, indent=2), encoding='utf-8')

        with (FOLDED_DATASET_DIR / 'category_mapping.csv').open('w', encoding='utf-8-sig', newline='') as f:
            fieldnames = ['rfdetr_internal_label', 'coco_category_id', 'category_id', 'n_number', 'name', 'is_placeholder']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(category_rows)

        # Validate class 0 is never used in annotations.
        class0_annotation_count = 0
        for path in FOLDED_DATASET_DIR.glob('fold*/**/_annotations.coco.json'):
            payload = json.loads(path.read_text(encoding='utf-8'))
            class0_annotation_count += sum(1 for ann in payload.get('annotations', []) if int(ann['category_id']) == 0)
        if class0_annotation_count != 0:
            raise RuntimeError(f'class0 annotation count must be 0, got {class0_annotation_count}')

        ready = {
            'folds': fold_summaries,
            'records': len(records),
            'real_classes': len(real_category_ids),
            'class0_placeholder': bool(ADD_CLASS0_PLACEHOLDER),
            'valid_include_canvas': bool(VALID_INCLUDE_CANVAS),
            'class0_annotation_count': class0_annotation_count,
            'label_map': str(FOLDED_DATASET_DIR / 'label_map_74.json'),
            'category_mapping': str(FOLDED_DATASET_DIR / 'category_mapping.csv'),
        }
        ready_marker.write_text(json.dumps(ready, ensure_ascii=False, indent=2), encoding='utf-8')
        print(json.dumps(ready, ensure_ascii=False, indent=2))
else:
    print('Skipping 5-fold dataset build. Using existing:', FOLDED_DATASET_DIR)


In [ ]:
# Create per-fold Colab CUDA configs from the repo config
import copy
import yaml

base_config_path = RFDETR_DIR / BASE_CONFIG_NAME
if not base_config_path.exists():
    base_config_path = RFDETR_DIR / 'config_74_hidden45.yaml'
print('base config:', base_config_path)

base_config = yaml.safe_load(base_config_path.read_text(encoding='utf-8'))

TRAIN_AUG_CONFIG = {
    'RandomBrightnessContrast': {
        'brightness_limit': float(AUG_BRIGHTNESS_LIMIT),
        'contrast_limit': float(AUG_CONTRAST_LIMIT),
        'p': float(AUG_COLOR_P),
    },
    'Affine': {
        'scale': [float(AUG_SCALE_RANGE[0]), float(AUG_SCALE_RANGE[1])],
        'keep_ratio': True,
        'translate_percent': [float(AUG_TRANSLATE_PERCENT[0]), float(AUG_TRANSLATE_PERCENT[1])],
        'rotate': [-float(AUG_ROTATE_LIMIT_DEGREES), float(AUG_ROTATE_LIMIT_DEGREES)],
        'shear': 0,
        'border_mode': 4,
        'p': float(AUG_AFFINE_P),
    },
}

if REQUIRE_DRIVE_FOR_CHECKPOINTS and not Path('/content/drive/MyDrive').exists():
    raise RuntimeError('Drive is not mounted; refusing to create checkpoint/output dirs outside persistent Drive.')

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
fold_config_paths = []
fold_model_tags = []

for fold_idx in FOLD_INDICES:
    fold_dir = FOLDED_DATASET_DIR / f'fold{fold_idx}'
    if not (fold_dir / 'train' / '_annotations.coco.json').exists():
        raise FileNotFoundError(f'fold train annotations missing: {fold_dir}')
    if not (fold_dir / 'valid' / '_annotations.coco.json').exists():
        raise FileNotFoundError(f'fold valid annotations missing: {fold_dir}')

    config = copy.deepcopy(base_config)
    config['data']['dataset_dir'] = str(fold_dir)
    config['data']['base56_dir'] = ''
    config['data']['hidden18_dir'] = ''
    config['data']['test_images_dir'] = ''

    fold_tag = f'{MODEL_TAG}_p{fold_idx:02d}'
    config['model']['variant'] = MODEL_VARIANT
    config['model']['tag'] = fold_tag

    train_cfg = config['train']
    train_cfg.update({
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'grad_accum_steps': GRAD_ACCUM_STEPS,
        'seed': 42 + int(fold_idx),
        'device': 'cuda',
        'pin_memory': PIN_MEMORY,
        'num_workers': NUM_WORKERS,
        'persistent_workers': bool(NUM_WORKERS > 0),
        'prefetch_factor': 2 if NUM_WORKERS > 0 else None,
        'checkpoint_interval': 1,
        'auto_resume': True,
        'resume': None,
        'eval_interval': 1,
        'early_stopping': EARLY_STOPPING,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'early_stopping_min_delta': EARLY_STOPPING_MIN_DELTA,
        'early_stopping_use_ema': EARLY_STOPPING_USE_EMA,
        'eval_max_dets': 100,
        'compute_val_loss': True,
        'compute_test_loss': False,
        'run_test': False,
        'progress_bar': 'tqdm',
        'tensorboard': True,
        'multi_scale': ENABLE_MULTI_SCALE,
        'expanded_scales': False,
        'amp_dtype': 'auto',
    })
    if ENABLE_TRAIN_AUGMENTATION:
        train_cfg['aug_config'] = TRAIN_AUG_CONFIG
        train_cfg['augmentation_backend'] = 'cpu'
    else:
        train_cfg.pop('aug_config', None)
        train_cfg['augmentation_backend'] = 'cpu'

    config['output']['local_output_dir'] = str(OUTPUT_ROOT)
    config['output']['backup_dir'] = str(BACKUP_DIR)

    config_path = RFDETR_DIR / f'{CONFIG_STEM}_p{fold_idx:02d}.yaml'
    config_path.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')
    fold_config_paths.append(config_path)
    fold_model_tags.append(fold_tag)
    print('wrote config:', config_path)
    print('fold tag:', fold_tag)
    print('fold dataset:', fold_dir)

print('model variant:', MODEL_VARIANT)
print('rfdetr min version:', RFDETR_MIN_VERSION)
print('folds:', FOLD_INDICES)
print('train augmentation enabled:', ENABLE_TRAIN_AUGMENTATION)
if ENABLE_TRAIN_AUGMENTATION:
    print(yaml.safe_dump(TRAIN_AUG_CONFIG, allow_unicode=True, sort_keys=False))
print('checkpoints will be saved under:', OUTPUT_ROOT / '<fold_model_tag>')
print('best checkpoint backups will be saved under:', BACKUP_DIR)


In [ ]:
# Preflight: validates imports, config, dataset files, and train kwargs without training
if RUN_IMPORT_SMOKE_TEST:
    smoke_code = f"""
import importlib.metadata
import sys
from packaging.version import Version
print('smoke: python', sys.executable, flush=True)
import torch
print('smoke: torch', torch.__version__, 'cuda_available=', torch.cuda.is_available(), flush=True)
if torch.cuda.is_available():
    print('smoke: cuda device', torch.cuda.get_device_name(0), flush=True)
import rfdetr
version = importlib.metadata.version('rfdetr')
print('smoke: rfdetr', version, 'RFDETRLarge=', hasattr(rfdetr, 'RFDETRLarge'), flush=True)
if Version(version) < Version('{RFDETR_MIN_VERSION}'):
    raise RuntimeError('rfdetr version is below required minimum')
if not hasattr(rfdetr, 'RFDETRLarge'):
    raise RuntimeError('RFDETRLarge is unavailable')
""".strip()
    run([sys.executable, '-u', '-c', smoke_code], cwd=RFDETR_DIR, heartbeat_seconds=15)
else:
    print('Skipping import smoke test.')

if RUN_DRY_RUN:
    for config_path in fold_config_paths:
        print('dry-run config:', config_path)
        run([sys.executable, '-u', 'train_74_hidden45.py', '--config', config_path, '--dry-run'], cwd=RFDETR_DIR, heartbeat_seconds=15)
else:
    print('Skipping dry-run.')


In [ ]:
# Full 5-fold train / resume / finalize
for fold_idx, config_path in zip(FOLD_INDICES, fold_config_paths):
    train_cmd = [sys.executable, '-u', 'train_74_hidden45.py', '--config', config_path]
    if TRAIN_EPOCHS_OVERRIDE is not None:
        train_cmd += ['--epochs', str(TRAIN_EPOCHS_OVERRIDE)]

    finalize_cmd = [sys.executable, '-u', 'train_74_hidden45.py', '--config', config_path, '--finalize-only']
    print('=' * 80)
    print(f'fold {fold_idx}')
    print('train command:', ' '.join(map(str, train_cmd)))
    print('finalize command:', ' '.join(map(str, finalize_cmd)))

    if RUN_FINALIZE_ONLY:
        run(finalize_cmd, cwd=RFDETR_DIR)
    elif RUN_FULL_TRAIN:
        run(train_cmd, cwd=RFDETR_DIR)
    else:
        print('RUN_FULL_TRAIN=False. Set RUN_FULL_TRAIN=True in the parameter cell to launch or resume this fold.')


In [ ]:
# Metrics and checkpoints for all selected folds
import json
import pandas as pd

for fold_idx, fold_tag in zip(FOLD_INDICES, fold_model_tags):
    output_dir = OUTPUT_ROOT / fold_tag
    backup_dir = BACKUP_DIR
    metrics_csv = output_dir / 'metrics.csv'
    print('=' * 80)
    print('fold:', fold_idx)
    print('output_dir:', output_dir)
    print('backup_dir:', backup_dir)
    print('metrics_csv:', metrics_csv)

    if output_dir.exists():
        for path in sorted(output_dir.glob('*')):
            size = f'{path.stat().st_size / (1024 * 1024):.1f} MB' if path.is_file() else '<dir>'
            print(path.name, size)
    else:
        print('No output_dir yet.')

    if metrics_csv.exists():
        metrics_df = pd.read_csv(metrics_csv)
        cols = [c for c in ['epoch', 'val/mAP_75', 'val/mAP_50_95', 'val/mAP_50', 'val/mAR', 'val/loss', 'train/loss'] if c in metrics_df.columns]
        per_epoch = metrics_df.groupby('epoch').last().reset_index()[cols]
        display(per_epoch.tail(20))
        if 'val/mAP_75' in per_epoch.columns and per_epoch['val/mAP_75'].notna().any():
            best_idx = per_epoch['val/mAP_75'].idxmax()
            best = per_epoch.loc[best_idx]
            print(f"Best mAP@0.75: epoch={int(best['epoch'])}, score={float(best['val/mAP_75']):.6f}")
            print('best mAP75 checkpoint:', output_dir / 'checkpoint_best_map75.ckpt')
    else:
        print('metrics.csv not found yet. Run training first.')

    map75_summary_path = output_dir / 'map75_summary.json'
    if map75_summary_path.exists():
        print('map75 summary:')
        print(map75_summary_path.read_text(encoding='utf-8'))

label_map_path = FOLDED_DATASET_DIR / 'label_map_74.json'
category_mapping_path = FOLDED_DATASET_DIR / 'category_mapping.csv'
print('label_map_74:', label_map_path, 'exists=', label_map_path.exists())
print('category_mapping:', category_mapping_path, 'exists=', category_mapping_path.exists())


## Local Submission/Postprocess

This Colab notebook trains RF-DETR Large 5-fold checkpoints only.

After training, download or sync the fold checkpoints from Drive and run the local inference notebook/script for:

- fold ensemble inference
- class `0` dummy prediction drop
- final category-id remap via `label_map_74.json` / `category_mapping.csv`
- submission filtering policy selection
- low-confidence image review


In [ ]:
# No Colab submission in this notebook
print('Submission/postprocess is intentionally skipped here. Use local inference after checkpoints are ready.')
print('Fold model tags:')
for fold_tag in fold_model_tags:
    print(' -', fold_tag)
print('Fold label map:', FOLDED_DATASET_DIR / 'label_map_74.json')
print('Fold category mapping:', FOLDED_DATASET_DIR / 'category_mapping.csv')


In [ ]:
# Reserved for local-postprocess notes
print('No-op: final submission CSV is generated locally, not in Colab.')


In [ ]:
# Reserved for local-postprocess sanity checks
print('No-op: local postprocess should drop category_id=0 and then apply the selected box filtering policy.')


In [ ]:
# Optional TensorBoard
if RUN_SHOW_TENSORBOARD:
    ip = get_ipython()
    ip.run_line_magic('load_ext', 'tensorboard')
    ip.run_line_magic('tensorboard', f'--logdir {OUTPUT_ROOT}')
else:
    print('Skipping TensorBoard. Set RUN_SHOW_TENSORBOARD=True to show it.')


## Augmentation And Class-0 Notes

This Large 5-fold notebook keeps the same train-only augmentation profile as the recent nano Colab run:

```python
ENABLE_TRAIN_AUGMENTATION = True
AUGMENTATION_NAME = 'aug_scale150_rot90_v1'
MODEL_VARIANT = 'large'
MODEL_TAG = f'{BASE_MODEL_TAG}_{AUGMENTATION_NAME}'
```

Current train-only augmentation:

- scale `[0.85, 1.50]`
- translate `[-0.04, 0.04]`
- rotate `[-90, 90]` degrees via `AUG_ROTATE_LIMIT_DEGREES`
- brightness/contrast `0.08`, probability `0.25`
- no flip, because mirrored pill imprints are unrealistic

Validation folds exclude synthetic `canvas__` images by default (`VALID_INCLUDE_CANVAS=False`), while train folds may still use canvas samples.

Class `0` is included as the background/no-object dummy category. It is not a real pill class and should not receive bbox annotations; RF-DETR/DETR learns background through unmatched queries and non-annotated image regions. The fold builder validates that `category_id=0` has zero annotations. Local inference/postprocess should drop any prediction mapped to category `0`.

Training is capped at `EPOCHS=100`, with early stopping enabled (`patience=1`, `min_delta=0.001`, `use_ema=True`). Because `eval_interval=1`, patience is counted by validation epochs/evaluations rather than mini-batches. One epoch is roughly 2.3k train images per fold, so this is the closest match to a ~2k-image no-improvement window.

Checkpoint safety remains on: `checkpoint_interval=1`, `auto_resume=True`, and all outputs are written under Drive `OUTPUT_ROOT / fold_model_tag`; best mAP@0.75 copies go to `BACKUP_DIR` with the fold model tag in the filename.


## Expected Outputs

After a completed full 5-fold run, outputs are isolated by fold-specific model tags such as:

```text
large_74_hidden45_canvas_balanced_5fold_cls0_colab_cuda_rfdetr_v18plus_aug_scale150_rot90_v1_p00
..._p01
..._p02
..._p03
..._p04
```

Per fold, expect:

- `metrics.csv` under `OUTPUT_ROOT / fold_model_tag`
- RF-DETR regular best checkpoint: `checkpoint_best_total.pth`
- wrapper-selected best mAP@0.75 checkpoint: `checkpoint_best_map75.ckpt`
- Drive backup copies under `BACKUP_DIR`
- `map75_summary.json` with the selected epoch and score

Dataset/cache/mapping artifacts:

- cached archive: `ARCHIVE_CACHE_DIR / DATASET_ARCHIVE_NAME` when archive download/copy is needed
- extracted source dataset: `DATASET_DIR`
- `FOLDED_DATASET_DIR / fold0..fold4 / {train,valid}`
- `FOLDED_DATASET_DIR / label_map_74.json`
- `FOLDED_DATASET_DIR / category_mapping.csv`
- `FOLDED_DATASET_DIR / _5fold_ready.json`
- fold summaries include `valid_canvas_images`, which should be `0` when `VALID_INCLUDE_CANVAS=False`

Training is launched with `python -u` and `PYTHONUNBUFFERED=1` so Colab logs should stream during long runs. If Colab disconnects, rerun the notebook. With `auto_resume: true`, each fold resumes from the latest `checkpoint_N.ckpt` in its Drive output folder. This fold4-only copy writes to the Colab/Drive output tag ending in `_p04`, separate from local MPS outputs.
